### Setup

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


### Load and Process DataFrame

In [ ]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
    rating_threshold=4.0, # replica 3.5
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 3.5
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5, threshold=5
)
interaction_info_df.head()

extracting item features...


merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[[RARE], [RARE], [RARE], [RARE], [RARE]]",USA,[RARE],Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,[RARE],Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [ ]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75, # replica 0.64
    val_ratio=0.10, # replica 0.16
    test_ratio=0.15, # replica 0.20
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.64 : 0.16 : 0.2):
train: 305206 (63.8%)
valid: 75578 (15.8%)
test: 97620 (20.41%)
------------------------------ 

Check target label distribution after splitting (%):
train label
1    0.649732
0    0.350268
Name: proportion, dtype: float64
valid label
1    0.595464
0    0.404536
Name: proportion, dtype: float64
test label
1    0.615745
0    0.384255
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 305206
  Num of positive interactions: 198302 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 396604], edge_label=[198302])
Edge Index: tensor([[    0,     0,     0,  ..., 10457, 10458, 10458],
        [ 2094,  2205,  2213,  ...,  1645,  1696,  1760]])


#### Evaluate User Diversity Preference Scale

In [8]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True)
user_dps_df.head(1)

Calculating user diversity preference scale:   0%|          | 0/2064 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:05<00:00, 381.94it/s]


,userID,actorID_wvec,actorID_dps,country_wvec,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.55008,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.110267,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.473433,"[74.5, 34.0, 8.0, 5.0, 35.5, 45.5, 0.0, 37.0, ...",0.773997


#### Prepare Item Multihot Vec on Each Dimension

In [9]:
item_vec_df = evaluator.get_item_feature_multihot_vec(encoded_train_df, feature_engineer.vocab2idx)
item_vec_df.head(1)

,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,1556,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


#### Prepare train/valid triplet data

In [10]:
train_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_train_df, k_negative_samples=5)
train_triplet_with_dps_df = train_triplet_df.merge(user_dps_df, on="userID", how="left")
train_triplet_with_dps_df = train_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="left")
# train_triplet_with_dps_df.head(1)

valid_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_valid_df, k_negative_samples=5)
valid_triplet_with_dps_df = valid_triplet_df.merge(user_dps_df, on="userID", how="left")
valid_triplet_with_dps_df = valid_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="inner")
valid_triplet_with_dps_df.head(1)

Original data count (positive samples): 198302
Num of triplets: 198302(pos samples) * 5(negative sampled items) = 991510
Original data count (positive samples): 45004
Num of triplets: 45004(pos samples) * 5(negative sampled items) = 225020


,userID,pos_item_id,neg_item_id,actorID_idx,country_idx,directorID_idx,genre_idx,neg_actorID_idx,neg_country_idx,neg_directorID_idx,...,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,0,4968,3608,"[334, 1644, 474, 755, 1206]",37,51,"[2, 9, 18, 0, 0, 0, 0, 0]","[2163, 337, 688, 1, 1]",37,447,...,0.110267,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.473433,"[74.5, 34.0, 8.0, 5.0, 35.5, 45.5, 0.0, 37.0, ...",0.773997,4968,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."


#### Prepare prediction pool for inference/testing

In [11]:
# NOTE: Prepare prediction pool to evaluate the model
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=1000)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2064
Item Pool: 7146, negative sampled to 1000 items for each user
Num of interactions: 2064(users) * 1000(items) = 2064000


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
2063995,2063,1961,0,"[470, 1508, 739, 1654, 681]",37,104,"[2, 19, 0, 0, 0, 0, 0, 0]"
2063996,2063,367,0,"[1257, 2020, 270, 1360, 1]",37,1,"[9, 15, 16, 18, 0, 0, 0, 0]"
2063997,2063,6867,0,"[1, 1, 1, 1, 1]",21,27,"[9, 10, 0, 0, 0, 0, 0, 0]"
2063998,2063,4258,0,"[246, 1972, 2178, 2094, 2263]",37,448,"[9, 18, 0, 0, 0, 0, 0, 0]"
2063999,2063,1138,0,"[2146, 2189, 2121, 1, 1]",37,1,"[7, 0, 0, 0, 0, 0, 0, 0]"


### Prepare DataLoader

In [12]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import TripletDataset, UserItemPairDataset, UserPosItemSampler, get_user_triplet_mapping

BATCH_SIZE = 1024

train_dataset = TripletDataset(train_triplet_with_dps_df)
valid_dataset = TripletDataset(valid_triplet_with_dps_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

# TODO: Custom sampler
# MIN_POS_ITEMS = 2
# user_to_pos_items, user_pos_to_indices = get_user_triplet_mapping(train_triplet_with_dps_df, MIN_POS_ITEMS)
# train_sampler = UserPosItemSampler(user_to_pos_items, user_pos_to_indices, batch_size=BATCH_SIZE, min_pos_items=MIN_POS_ITEMS, max_pos_items=20, buffer=0)
# train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=4, worker_init_fn=seed_worker, generator=g)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 991510
valid data count: 223695
test data count: 2064000


### Configure Model (LightningModule)

In [13]:
from lightning_models.mtdp_ngcf_v2 import MTDPRecSRM

EMB_DIM = 64
LR = 1e-3
EPOCHS = 50
NUM_LAYERS = 3
REG_WEIGHT = 1e-5

DPS_WEIGHTS = {
    "actor_dps": 0.25,
    "country_dps": 0.25,
    "director_dps": 0.25,
    "genre_dps": 0.25,
}

DPR_WEIGHTS = {
    "actor_dpr": 0.25,
    "country_dpr": 0.25,
    "director_dpr": 0.25,
    "genre_dpr": 0.25,
}

DPM_WEIGHTS = {
    "actor_pd": 0.25,
    "country_pd": 0.25,
    "director_pd": 0.25,
    "genre_pd": 0.25,
}

MT_WEIGHTS = {
    "bpr_loss": 1.0,
    "dps_loss": 0,
    "dpr_loss": 0,
    "dpm_loss": 0,
}

RESCALE_METHOD = None  # None, "log", "ema"

model = MTDPRecSRM(
    graph_data=train_graph,  # shape [2, num_edges]
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    embedding_dim=EMB_DIM,
    num_layers=NUM_LAYERS,
    node_dropout=0.0,
    mess_dropout=0.1,
    lr=LR,
    reg_weight=REG_WEIGHT,
    dps_weights=DPS_WEIGHTS,
    dpr_weights=DPR_WEIGHTS,
    dpm_weights=DPM_WEIGHTS,
    mt_weights=MT_WEIGHTS,
    rescale_method=RESCALE_METHOD,
)


Seed set to 42


### Configure Trainer and Experiment

In [ ]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "rerank"
VERSION = "ngcf_v2"
RUN_NAME = "run01(K=1000)"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, tags={"version": VERSION})
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    version_name=VERSION,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_loss",
    monitor_mode="min",
    hyper_param_str=f"",
)

In [15]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [16]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/adam/R11_Bai/DPRecSys/experiments/test_checkpoints/rerank exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

   | Name                | Type              | Params | Mode 
-------------------------------------------------------------------
0  | ngcf_model          | NGCF              | 694 K  | train
1  | bpr_loss            | BPRLoss           | 0      | train
2  | reg_loss            | EmbLoss           | 0      | train
3  | dps_module          | DPSPredictor      | 1.0 K  | train
4  | dps_loss_fn         | DPSLoss           | 0      | train
5  | dpr_module          | DPRegularizer     | 263 K  | train
6  | dpr_loss_fn         | DPRLoss           | 0      | train
7  | dpm_module          | DPMatcher         | 0      | train
8  | dpm_loss_fn         | KLDivergenceLoss  | 0      | train
9  | loss_scaling_module | LogScaleLoss  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 4, global step 4845: 'val_loss' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 5 records. Best score: 0.575. Signaling Trainer to stop.
Epoch 5, global step 5814: 'val_loss' was not in top 1


🏃 View run run02 at: http://140.112.106.216:3683/#/experiments/19/runs/100804b66da140bdaa776fdc130208f6
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/19


### Inference

In [17]:
# NOTE: the inference model MUST be the same as the training model
# best_model_experiment_name = "rerank"
# best_model_checkpoint_path = "[ngcf_v2]-test_run--best-checkpoint-epoch=00-val_loss=2.93.ckpt"
# best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"
best_model_path = trainer.checkpoint_callback.best_model_path

model = MTDPRecSRM.load_from_checkpoint(checkpoint_path=best_model_path)
# start inference
trainer.test(model=model, dataloaders=test_loader)


Seed set to 42
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test_ndcg10        │    0.33384597301483154    │
│        test_ndcg20        │    0.3751211166381836     │
│        test_ndcg5         │    0.2780965268611908     │
│     test_precision10      │    0.1185562014579773     │
│     test_precision20      │    0.11400193721055984    │
│      test_precision5      │    0.11889535188674927    │
│       test_recall10       │    0.05762017145752907    │
│       test_recall20       │    0.10615308582782745    │
│       test_recall5        │   0.028827063739299774    │
└───────────────────────────┴───────────────────────────┘

🏃 View run run02 at: http://140.112.106.216:3683/#/experiments/19/runs/100804b66da140bdaa776fdc130208f6
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/19


[{'test_ndcg5': 0.2780965268611908,
  'test_ndcg10': 0.33384597301483154,
  'test_ndcg20': 0.3751211166381836,
  'test_precision5': 0.11889535188674927,
  'test_precision10': 0.1185562014579773,
  'test_precision20': 0.11400193721055984,
  'test_recall5': 0.028827063739299774,
  'test_recall10': 0.05762017145752907,
  'test_recall20': 0.10615308582782745}]

In [18]:
# exp 1000
model.test_results["eval_score_df"].describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.278097,0.028827,0.118895,0.333846,0.057620,0.118556,0.375121,0.106153,0.114002
std,595.969798,0.355372,0.056329,0.163670,0.314631,0.085705,0.130213,0.265450,0.120563,0.105898
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.231378,0.020000,0.050000
50%,1031.500000,0.000000,0.000000,0.000000,0.333333,0.029412,0.100000,0.376927,0.073171,0.100000
75%,1547.250000,0.543771,0.038462,0.200000,0.558456,0.081633,0.200000,0.545018,0.153846,0.150000
max,2063.000000,1.000000,1.000000,0.800000,1.000000,1.000000,0.600000,1.000000,1.000000,0.600000


In [19]:
# model.test_results["eval_score_df"].describe()

In [ ]:
# Evaluate user diversity preference matching score (DPMS) at k
eval_df = evaluator.prepare_evaluation_data(model.test_results, feature_engineer.idx2vocab)

# Get user DPMS
user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

user_dpms_df.describe()

candidate item pool size: 20
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 41280
interaction data count after merging: 41280
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:03<00:00, 658.59it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.168444,0.977181,0.173344,0.877976,0.549236
std,595.969798,0.082508,0.035907,0.116330,0.076633,0.059093
min,0.000000,0.000000,0.328237,0.000000,0.224334,0.277679
25%,515.750000,0.104403,0.973591,0.083938,0.846258,0.510069
50%,1031.500000,0.170069,0.986814,0.163671,0.898514,0.550550
75%,1547.250000,0.228086,0.993105,0.252670,0.931508,0.591542
max,2063.000000,0.406400,1.000000,0.560385,0.986570,0.717935


In [ ]:
prefix = "(mt)ngcf_v2_k5_1000"
embedding_path = "embeddings/"

torch.save(model.user_emb.cpu(), f"{embedding_path}{prefix}_user_emb.pt")
torch.save(model.item_emb.cpu(), f"{embedding_path}{prefix}_item_emb.pt")

In [ ]:
import joblib
dir = "artifacts"
prefix = "(mt)ngcf_v2_k5_1000"
if not os.path.exists(dir):
    os.makedirs(dir)

joblib.dump(eval_df, os.path.join(dir, f"{prefix}_eval_df.pkl"))
joblib.dump(user_dps_df, os.path.join(dir, f"{prefix}_user_dps_df.pkl"))
joblib.dump(feature_engineer, os.path.join(dir, f"{prefix}_feature_engineer.pkl"))

['artifacts/(mt)ngcf_v2_k5_replica_feature_engineer.pkl']

In [3]:
import os
import sys
import joblib

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

dir = "artifacts"
prefix = "(mt)ngcf_v2_k5_replica"
eval_df = joblib.load(os.path.join(dir, f"{prefix}_eval_df.pkl"))
user_dps_df = joblib.load(os.path.join(dir, f"{prefix}_user_dps_df.pkl"))
feature_engineer = joblib.load(os.path.join(dir, f"{prefix}_feature_engineer.pkl"))

In [4]:
from common.eval import Evaluator
evaluator = Evaluator()
# Evaluate user diversity preference matching score (DPMS) at k

# Get user DPMS
user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

user_dpms_df.describe()

Seed set to 42


candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:01<00:00, 1150.03it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.130237,0.962956,0.138186,0.833041,0.516105
std,595.969798,0.075820,0.047787,0.115240,0.090805,0.058524
min,0.000000,0.000000,0.301659,0.000000,0.149020,0.233691
25%,515.750000,0.072842,0.953539,0.041908,0.790179,0.476714
50%,1031.500000,0.127326,0.977777,0.125497,0.853099,0.516180
75%,1547.250000,0.180733,0.989389,0.208108,0.897550,0.556388
max,2063.000000,0.378228,1.000000,0.623538,0.978662,0.703713


### MMR

In [24]:
from post_processing.mmr import MMR

mmr_reranker = MMR(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    m=100,
)


Seed set to 42


In [25]:
mmr_result_df = mmr_reranker.rerank(
    top_k=10,
    theta=0.5,
    random_state=42,
)
mmr_result_df.head()

Preparing input DataFrame for MMR...
candidate item pool size: 100
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 206400
interaction data count after merging: 206400
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064
Reranking items for each user...


Reranking users: 100%|██████████| 2064/2064 [00:14<00:00, 146.31it/s]

Reranked DataFrame shape: (2064, 2)
Final Reranked DataFrame shape: (2064, 4)


,user,rec_items,asis_rec_items,gt_items
0,75,"[356, 4993, 2467, 5064, 3253, 588, 1233, 1252,...","[356, 318, 3147, 4993, 5418, 588, 1252, 7153, ...","[45722, 1233, 110, 2959, 2571]"
1,78,"[50872, 4766, 1090, 2916, 7587, 8807, 50274, 5...","[50872, 46578, 4903, 4306, 1221, 4766, 8807, 3...","[4119, 6993, 8400, 50872]"
2,127,"[42723, 41716, 52952, 37382, 26425, 8025, 3429...","[42723, 41863, 26425, 41716, 52975, 37382, 576...","[45726, 6958]"
3,170,"[296, 1348, 1090, 1291, 39183, 4011, 1965, 508...","[296, 1291, 1348, 1200, 1965, 260, 2353, 4011,...","[1222, 3949, 4011, 2542, 8874, 44191, 45728, 3..."
4,175,"[7387, 2677, 5995, 2174, 49772, 8225, 40414, 4...","[7387, 5995, 2174, 2677, 8225, 1674, 25788, 40...","[1913, 1419, 4927, 50068, 7700, 1757, 5300, 51..."


In [26]:
from common.eval import Evaluator
evaluator = Evaluator()

mmr_reranked_score_df = evaluator.evaluate(mmr_result_df, K=5)
mmr_reranked_score_df = evaluator.evaluate(mmr_reranked_score_df, K=10)
mmr_reranked_score_df = evaluator.evaluate(mmr_reranked_score_df, K=20)
mmr_reranked_score_df.describe()

Seed set to 42


,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.294215,0.055156,0.122771,0.342781,0.095247,0.111870,0.342781,0.095247,0.055935
std,20797.975208,0.368147,0.105437,0.166934,0.331342,0.133062,0.126282,0.331342,0.133062,0.063141
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35054.000000,0.000000,0.000000,0.000000,0.333333,0.055556,0.100000,0.333333,0.055556,0.050000
75%,53331.000000,0.619729,0.080000,0.200000,0.578454,0.142857,0.200000,0.578454,0.142857,0.100000
max,71534.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.500000


In [27]:
reranked_user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=mmr_result_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

reranked_user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 722.20it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.105912,0.838454,0.107156,0.859096,0.477655
std,595.969798,0.062720,0.097294,0.091695,0.076098,0.053813
min,0.000000,0.000000,0.387094,0.000000,0.304835,0.232462
25%,515.750000,0.056976,0.773805,0.029505,0.822298,0.442686
50%,1031.500000,0.103230,0.852554,0.097585,0.877755,0.479890
75%,1547.250000,0.149038,0.913959,0.162470,0.912019,0.515057
max,2063.000000,0.381907,0.998682,0.561380,0.980987,0.689921


### DPA-RS

In [28]:
from post_processing.dpa_rs import DPA_RS

reranker = DPA_RS(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    m=100,
    ground_truth_dps_df=user_dps_df
)


Seed set to 42


In [29]:
result_df = reranker.rerank(
    top_k=10,
    max_iter=100,
    random_state=42,
)
result_df.head()

Preparing input DataFrame for DPA-RS...
candidate item pool size: 100
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 206400
interaction data count after merging: 206400
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064
Combined DataFrame shape: (206400, 16)
Reranking items for each user...


Reranking users: 100%|██████████| 2064/2064 [10:53<00:00,  3.16it/s]

Reranked DataFrame shape: (2064, 2)
Final Reranked DataFrame shape: (2064, 4)


,user,rec_items,asis_rec_items,gt_items
0,75,"[50442, 733, 1839, 6683, 27728, 2325, 6294, 35...","[356, 318, 3147, 4993, 5418, 588, 1252, 7153, ...","[45722, 1233, 110, 2959, 2571]"
1,78,"[46578, 27660, 8601, 3340, 52717, 5015, 6065, ...","[50872, 46578, 4903, 4306, 1221, 4766, 8807, 3...","[4119, 6993, 8400, 50872]"
2,127,"[5187, 4639, 42723, 40583, 7157, 5128, 8865, 3...","[42723, 41863, 26425, 41716, 52975, 37382, 576...","[45726, 6958]"
3,170,"[296, 50, 420, 761, 3897, 8917, 1265, 5450, 46...","[296, 1291, 1348, 1200, 1965, 260, 2353, 4011,...","[1222, 3949, 4011, 2542, 8874, 44191, 45728, 3..."
4,175,"[52375, 8917, 33817, 53996, 1608, 2557, 1176, ...","[7387, 5995, 2174, 2677, 8225, 1674, 25788, 40...","[1913, 1419, 4927, 50068, 7700, 1757, 5300, 51..."


In [30]:
from common.eval import Evaluator
evaluator = Evaluator()

reranked_score_df = evaluator.evaluate(result_df, K=5)
reranked_score_df = evaluator.evaluate(reranked_score_df, K=10)
reranked_score_df = evaluator.evaluate(reranked_score_df, K=20)
reranked_score_df.describe()

Seed set to 42


,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.201076,0.034339,0.080717,0.260696,0.068190,0.081686,0.260696,0.068190,0.040843
std,20797.975208,0.318281,0.079694,0.131128,0.295939,0.112350,0.103426,0.295939,0.112350,0.051713
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35054.000000,0.000000,0.000000,0.000000,0.289065,0.020408,0.100000,0.289065,0.020408,0.050000
75%,53331.000000,0.430677,0.040000,0.200000,0.441307,0.100000,0.100000,0.441307,0.100000,0.050000
max,71534.000000,1.000000,1.000000,0.800000,1.000000,1.000000,0.600000,1.000000,1.000000,0.300000


In [31]:
reranked_user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=result_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

reranked_user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 711.25it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.255917,0.975943,0.410577,0.930374,0.643202
std,595.969798,0.079787,0.026074,0.142490,0.031427,0.046939
min,0.000000,0.000000,0.700477,0.000000,0.747018,0.475511
25%,515.750000,0.205172,0.970482,0.335812,0.914246,0.616393
50%,1031.500000,0.253736,0.983414,0.420300,0.935136,0.645742
75%,1547.250000,0.305357,0.991395,0.500753,0.953217,0.674001
max,2063.000000,0.639933,1.000000,0.899103,0.991833,0.836592
